# GRAM Recurrent Qwen - SVGD Resume

Fresh Colab notebook for the SVGD particle update branch.

This notebook expects you to upload the rebuilt project zip:

`gram-recurrent-qwen-colab-upload.zip`

It does not embed checkpoints. It mounts Google Drive and restores checkpoints/data by filename. It also authenticates Hugging Face from Colab Secrets before any Hub downloads.

## 0. Runtime

Use a fresh GPU runtime. A100/H100 is ideal, but this smoke run should also fit smaller GPUs. The notebook wipes and recreates `/content/gram-recurrent-qwen` so stale imports do not leak into the run.

In [ ]:
import os, sys, subprocess
from pathlib import Path

print('python', sys.version)
subprocess.run(['nvidia-smi'], check=False)

## 1. Upload And Extract Project Zip

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
zip_name = next(k for k in uploaded if k.endswith('.zip'))

PROJECT = Path('/content/gram-recurrent-qwen')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)

!unzip -q {zip_name} -d /content/gram-recurrent-qwen
%cd /content/gram-recurrent-qwen
!find . -maxdepth 2 -type f | sort | head -120

## 2. Install Dependencies And Verify HF Auth

This uses `HF_TOKEN` from Colab Secrets. In Colab, open the key icon in the left sidebar and add a secret named `HF_TOKEN`.

In [ ]:
%cd /content/gram-recurrent-qwen
!pip -q install -r requirements.txt
!pip -q install -U huggingface_hub sentencepiece safetensors

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

token = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
assert token, 'Set HF_TOKEN in Colab Secrets before running this cell.'

os.environ['HF_TOKEN'] = token
os.environ['HUGGINGFACE_HUB_TOKEN'] = token
login(token=token, add_to_git_credential=False)
who = HfApi(token=token).whoami()
print('HF auth OK:', who.get('name') or who.get('email') or 'authenticated user')

## 3. Restore Drive Checkpoints And Data

Required checkpoint:

- `phase1_step_150.pt` -> `outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt`

Dataset files are restored if found, otherwise regenerated from Hugging Face.

In [ ]:
%cd /content/gram-recurrent-qwen

from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')

def restore_by_name(filename, dst, required=True, preferred_substring=None):
    dst = Path(dst)
    if dst.exists():
        print('exists:', dst)
        return dst
    matches = list(DRIVE_ROOT.rglob(filename))
    if preferred_substring:
        matches = sorted(matches, key=lambda p: preferred_substring not in str(p))
    if not matches:
        if required:
            raise FileNotFoundError(f'Could not find {filename} under {DRIVE_ROOT}')
        print('missing optional:', filename)
        return None
    src = matches[0]
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print('restored:', src, '->', dst)
    return dst

PHASE1_CKPT = 'outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt'
restore_by_name('phase1_step_150.pt', PHASE1_CKPT, preferred_substring='qwen_0_5b_phase1_a100_beta008_continue_150')

restore_by_name('opus47_train.jsonl', 'data/opus47_train.jsonl', required=False)
restore_by_name('opus47_val.jsonl', 'data/opus47_val.jsonl', required=False)

if not Path('data/opus47_train.jsonl').exists() or not Path('data/opus47_val.jsonl').exists():
    !python training/prepare_hf_reasoning_jsonl.py \
      --dataset_id lordx64/reasoning-distill-opus-4-7-max-sft \
      --tokenizer_name Qwen/Qwen2.5-0.5B-Instruct \
      --output_jsonl data/opus47_train.jsonl \
      --val_jsonl data/opus47_val.jsonl \
      --limit 1000 \
      --max_total_tokens 2048

!wc -l data/opus47_train.jsonl data/opus47_val.jsonl
!ls -lh outputs/qwen_0_5b_phase1_a100_beta008_continue_150

## 4. Sanity Gates

Local unit tests include the SVGD particle update and grouped trajectory continuation path. The identity gate uses the strict `float32 + eager` path.

In [ ]:
%cd /content/gram-recurrent-qwen
!pytest -q tests

!python eval/eval_identity.py \
  --model_name Qwen/Qwen2.5-0.5B-Instruct \
  --split 6,18 \
  --dtype float32 \
  --attn_implementation eager \
  --device cuda \
  --threshold 1e-3 | tee identity_gate.log

## 5. Validate Phase 1 Baseline

In [ ]:
%cd /content/gram-recurrent-qwen

!python eval/eval_jsonl.py \
  --model_name Qwen/Qwen2.5-0.5B-Instruct \
  --data_jsonl data/opus47_val.jsonl \
  --checkpoint outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt \
  --split 6,18 \
  --max_loops 4 \
  --max_length 512 \
  --beta 0.08 \
  --dtype bfloat16 \
  --adapter_dtype float32 \
  --device cuda | tee phase1_baseline_val.log

## 6. Train Phase 2 SVGD Smoke

This is a 25-step smoke run from the Phase 1 checkpoint. It is meant to prove the SVGD path is finite and produces measurable particle diversity, not to be research-quality training yet.

In [ ]:
%cd /content/gram-recurrent-qwen

!python training/train_phase2_stochastic.py \
  --config config/qwen_0_5b_phase2_svgd.yaml \
  --train_jsonl data/opus47_train.jsonl \
  --device cuda | tee phase2_svgd_smoke25_train.log

## 7. Validate Phase 2 SVGD

In [ ]:
%cd /content/gram-recurrent-qwen

!python eval/eval_jsonl.py \
  --model_name Qwen/Qwen2.5-0.5B-Instruct \
  --data_jsonl data/opus47_val.jsonl \
  --checkpoint outputs/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt \
  --split 6,18 \
  --max_loops 4 \
  --num_trajectories 4 \
  --particle_update_mode svgd \
  --particle_init_noise 0.02 \
  --svgd_repulsion_scale 0.5 \
  --svgd_repulsion_max_norm 1.0 \
  --max_length 512 \
  --beta 0.08 \
  --rho 1e-3 \
  --dtype bfloat16 \
  --adapter_dtype float32 \
  --device cuda | tee phase2_svgd_smoke25_val.log

## 8. Phase 1 vs Phase 2 Best-of-K

In [ ]:
%cd /content/gram-recurrent-qwen

!python eval/eval_best_of_k_jsonl.py \
  --phase1_checkpoint outputs/qwen_0_5b_phase1_a100_beta008_continue_150/phase1_step_150.pt \
  --phase2_checkpoint outputs/qwen_0_5b_phase2_svgd_smoke25/phase2_step_25.pt \
  --phase2_num_trajectories 4 \
  --phase2_particle_update_mode svgd \
  --particle_init_noise 0.02 \
  --svgd_repulsion_scale 0.5 \
  --svgd_repulsion_max_norm 1.0 \
  --max_new_tokens 64 \
  --dtype bfloat16 \
  --adapter_dtype float32 \
  --device cuda | tee phase1_vs_phase2_svgd_bestofk.log

## 9. Persist New Run To Drive

Run this after the smoke train/eval if the logs look sane. It copies the new SVGD checkpoint and logs to Drive so a runtime reset does not lose them.

In [ ]:
%cd /content/gram-recurrent-qwen

from pathlib import Path
from datetime import datetime
import shutil

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DST = Path('/content/drive/MyDrive/gram-recurrent-qwen-runs') / f'svgd_smoke_{stamp}'
RUN_DST.mkdir(parents=True, exist_ok=True)

for path in [
    Path('outputs/qwen_0_5b_phase2_svgd_smoke25'),
    Path('phase1_baseline_val.log'),
    Path('phase2_svgd_smoke25_train.log'),
    Path('phase2_svgd_smoke25_val.log'),
    Path('phase1_vs_phase2_svgd_bestofk.log'),
    Path('identity_gate.log'),
]:
    if path.exists():
        dst = RUN_DST / path.name
        if path.is_dir():
            shutil.copytree(path, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(path, dst)
        print('saved:', path, '->', dst)

print('run saved to:', RUN_DST)